# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marrwan1/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

My lane: Refresh / Content Opportunity Scoring

Rule idea:
A page is a refresh opportunity if it has high impressions but low CTR
relative to its position. Google is already showing the page but users
are not clicking — a content or title refresh could fix that.

Signal 1 (flag-linked): CTR vs Position
Same signal behind FlyRank's CTR-fix flag. Pages in positions 1-10
should have higher CTR — if they don't, that's the opportunity.
Verdict: CONFIRMED

Signal 2: Impressions Volume
High-impression pages have more to gain from a refresh — the audience
is there, the click opportunity is real.
Verdict: CONFIRMED

In [2]:
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")
MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Signal 1 — CTR vs Position (flag-linked)
signal1 = con.sql(f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 3  THEN '1_top3'
        WHEN gsc_avg_position <= 10 THEN '2_top10'
        WHEN gsc_avg_position <= 20 THEN '3_top20'
        ELSE                             '4_beyond20'
    END AS position_bucket,
    COUNT(*) AS n,
    ROUND(AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)), 4) AS avg_ctr
FROM read_parquet('{MAR}')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
GROUP BY position_bucket
ORDER BY position_bucket
""").df()

print("Signal 1 — CTR by Position Bucket")
print(signal1)
print("Verdict: CONFIRMED — higher position = higher CTR")

# Signal 2 — Impressions Volume
signal2 = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions < 10   THEN '1_low'
        WHEN gsc_impressions < 100  THEN '2_medium'
        WHEN gsc_impressions < 1000 THEN '3_high'
        ELSE                             '4_very_high'
    END AS impressions_bucket,
    COUNT(*) AS n,
    ROUND(AVG(gsc_clicks), 2) AS avg_clicks
FROM read_parquet('{MAR}')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
GROUP BY impressions_bucket
ORDER BY impressions_bucket
""").df()

print("\nSignal 2 — Clicks by Impressions Bucket")
print(signal2)
print("Verdict: CONFIRMED — higher impressions = higher clicks")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 — CTR by Position Bucket
  position_bucket        n  avg_ctr
0          1_top3   727362   0.0048
1         2_top10  1456122   0.0035
2         3_top20   519223   0.0028
3      4_beyond20   908354   0.0013
Verdict: CONFIRMED — higher position = higher CTR

Signal 2 — Clicks by Impressions Bucket
  impressions_bucket        n  avg_clicks
0              1_low  1463532        0.01
1           2_medium  1508921        0.10
2             3_high   606189        0.80
3        4_very_high    32419        5.07
Verdict: CONFIRMED — higher impressions = higher clicks


Rule: a page scores high if its actual CTR is below the expected CTR
for its position bucket, weighted by impressions volume.

Score     = (expected_ctr - actual_ctr) * gsc_impressions
Reason    = LOW_CTR_GOOD_POSITION or HIGH_IMP_LOW_CTR
Action    = REFRESH_TITLE_META or REFRESH_CONTENT

In [3]:
scored = con.sql(f"""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                         AS gsc_impressions,
        SUM(gsc_clicks)                                              AS gsc_clicks,
        AVG(gsc_avg_position)                                        AS gsc_avg_position,
        CASE WHEN SUM(gsc_impressions) = 0 THEN NULL
             ELSE SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        END                                                          AS ctr
    FROM read_parquet('{MAR}')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
),
expected_ctr AS (
    SELECT *,
        CASE
            WHEN gsc_avg_position <= 3  THEN 0.0048
            WHEN gsc_avg_position <= 10 THEN 0.0035
            WHEN gsc_avg_position <= 20 THEN 0.0028
            ELSE                             0.0013
        END AS expected_ctr
    FROM base
),
scored AS (
    SELECT *,
        -- Score: how much CTR is below expected, weighted by impressions
        ROUND((expected_ctr - ctr) * gsc_impressions, 2) AS opportunity_score,
        -- Reason code
        CASE
            WHEN gsc_avg_position <= 10 AND ctr < expected_ctr
                THEN 'LOW_CTR_GOOD_POSITION'
            WHEN gsc_impressions >= 1000 AND ctr < expected_ctr
                THEN 'HIGH_IMP_LOW_CTR'
            ELSE 'MONITOR'
        END AS reason_code,
        -- Action label
        CASE
            WHEN gsc_avg_position <= 10 AND ctr < expected_ctr
                THEN 'REFRESH_TITLE_META'
            WHEN gsc_impressions >= 1000 AND ctr < expected_ctr
                THEN 'REFRESH_CONTENT'
            ELSE 'NO_ACTION'
        END AS action
    FROM expected_ctr
)
SELECT * FROM scored
WHERE action != 'NO_ACTION'
ORDER BY opportunity_score DESC
""").df()

print(f"Queue size: {len(scored):,}")
scored.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue size: 88,556


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,expected_ctr,opportunity_score,reason_code,action
0,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,0.0035,719.41,LOW_CTR_GOOD_POSITION,REFRESH_TITLE_META
1,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,0.001420,0.0048,687.79,LOW_CTR_GOOD_POSITION,REFRESH_TITLE_META
2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,4.545582,0.000007,0.0035,471.44,LOW_CTR_GOOD_POSITION,REFRESH_TITLE_META
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,3.219473,0.000301,0.0035,457.57,LOW_CTR_GOOD_POSITION,REFRESH_TITLE_META
4,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,1.0,9.385150,0.000008,0.0035,433.26,LOW_CTR_GOOD_POSITION,REFRESH_TITLE_META
5,client_62f4a7e64f5e0096,content_7c6373141eae744a,132593.0,83.0,5.789019,0.000626,0.0035,381.08,LOW_CTR_GOOD_POSITION,REFRESH_TITLE_META
6,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,15.0,9.536301,0.000139,0.0035,361.54,LOW_CTR_GOOD_POSITION,REFRESH_TITLE_META
7,client_e547b89c05043229,content_306bc78dff1eb683,80821.0,35.0,1.488604,0.000433,0.0048,352.94,LOW_CTR_GOOD_POSITION,REFRESH_TITLE_META
8,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,0.003253,0.0048,342.29,LOW_CTR_GOOD_POSITION,REFRESH_TITLE_META
9,client_e547b89c05043229,content_9ef3d7516483e665,89229.0,92.0,2.481596,0.001031,0.0048,336.30,LOW_CTR_GOOD_POSITION,REFRESH_TITLE_META


In [6]:
import os
os.makedirs("work/outputs", exist_ok=True)

scored.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("✅ CSV written to work/outputs/baseline_action_score.csv")
print(f"   Rows: {len(scored):,}")

✅ CSV written to work/outputs/baseline_action_score.csv
   Rows: 88,556


Top-10 Review:

1. client_23a6 / content_44f3 | REFRESH_TITLE_META | 212k impressions, CTR=0.01% vs expected 0.35% | Wrong if: page is intentionally a no-click resource (e.g. a definition page)

2. client_e547 / content_8d7d | REFRESH_TITLE_META | 203k impressions, CTR=0.14% vs expected 0.48% | Wrong if: brand query with zero-click SERP feature stealing clicks

3. client_73cd / content_8e13 | REFRESH_TITLE_META | 135k impressions, 1 click only | Wrong if: page was recently published and hasn't settled yet

4. client_62f4 / content_34a7 | REFRESH_TITLE_META | 143k impressions, CTR=0.03% | Wrong if: position 3.2 but competing against rich snippets

5. client_73cd / content_fec5 | REFRESH_TITLE_META | 124k impressions, 1 click only | Wrong if: navigational query where users find answer in title

6. client_62f4 / content_7c63 | REFRESH_TITLE_META | 133k impressions, CTR=0.06% | Wrong if: informational query with featured snippet above

7. client_62f4 / content_f611 | REFRESH_TITLE_META | 108k impressions, CTR=0.01% | Wrong if: page targets a broad keyword with low commercial intent

8. client_e547 / content_306b | REFRESH_TITLE_META | 81k impressions, position=1.5, CTR=0.04% | Wrong if: SERP dominated by ads pushing organic results down

9. client_e547 / content_0e03 | REFRESH_TITLE_META | 221k impressions, CTR=0.33% vs expected 0.48% | Wrong if: CTR gap is too small to justify a refresh

10. client_e547 / content_9ef3 | REFRESH_TITLE_META | 89k impressions, CTR=0.10% | Wrong if: page already scheduled for refresh by the client

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Weak picks in my queue — cases where the rule fires but shouldn't:

1. Zero-click pages by design
   Pages that answer a query in the title/meta alone (definitions, dates,
   quick facts) will always have low CTR regardless of position. The rule
   flags them as opportunities but a refresh won't help.

2. Very low absolute clicks (1–3 clicks on 100k+ impressions)
   The CTR gap looks huge but the absolute signal is tiny — one extra click
   would have doubled the CTR. The opportunity score is inflated by
   impressions volume alone.

3. Pages dominated by SERP features
   If Google shows a featured snippet, People Also Ask, or an ad block above
   the organic result, low CTR is caused by the SERP layout, not the content.
   A refresh cannot fix that.

4. Recently published pages
   New pages haven't settled in rankings yet. Flagging them for refresh
   after one month of data is premature — they need more time.

Fix in Week 5: add a minimum clicks threshold and a page-age filter
before scoring.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.